In [1]:
!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign
!unzip -q gtsrb-german-traffic-sign.zip -d traffic_data

Dataset URL: https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign
License(s): CC0-1.0
100% 612M/612M [00:04<00:00, 155MB/s]



In [2]:
import os
import cv2
import numpy as np

data = []
labels = []
img_size = 32
train_path = "traffic_data/Train"

for class_folder in os.listdir(train_path):
    class_path = os.path.join(train_path, class_folder)
    for img_file in os.listdir(class_path):
        img = cv2.imread(os.path.join(class_path, img_file))
        img = cv2.resize(img, (img_size, img_size))
        data.append(img)
        labels.append(int(class_folder))

data = np.array(data) / 255.0
labels = np.array(labels)
print("Loaded", len(data), "images")

Loaded 39209 images


In [3]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

X_train, X_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, random_state=42)

num_classes = len(set(labels))
y_train = to_categorical(y_train, num_classes)
y_val = to_categorical(y_val, num_classes)

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model2 = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_size, img_size, 3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
history2 = model2.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=5, batch_size=32)

Epoch 1/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 45s 44ms/step - accuracy: 0.5231 - loss: 1.6847 - val_accuracy: 0.9135 - val_loss: 0.4114
Epoch 2/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 41s 42ms/step - accuracy: 0.8270 - loss: 0.5401 - val_accuracy: 0.9611 - val_loss: 0.1710
Epoch 3/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 42s 43ms/step - accuracy: 0.8903 - loss: 0.3410 - val_accuracy: 0.9719 - val_loss: 0.1184
Epoch 4/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 82s 43ms/step - accuracy: 0.9200 - loss: 0.2561 - val_accuracy: 0.9801 - val_loss: 0.0808
Epoch 5/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 82s 43ms/step - accuracy: 0.9336 - loss: 0.2110 - val_accuracy: 0.9850 - val_loss: 0.0645


In [7]:
loss2, acc2 = model2.evaluate(X_val, y_val)
print(f"Validation Accuracy: {acc2*100:.2f}%")

246/246 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9850 - loss: 0.0645
Validation Accuracy: 98.50%
